In [2]:
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Paths – adjust if your folder structure is different
DATA_DIR = Path("../data/dataset")  # or wherever your files are

train_s1 = pd.read_csv(DATA_DIR / "train/train_source1.tsv", sep="\t")
train_s2 = pd.read_csv(DATA_DIR / "train/train_source2.tsv", sep="\t")
train_s3 = pd.read_csv(DATA_DIR / "train/train_source3.tsv", sep="\t")
gt = pd.read_csv(DATA_DIR / "train/train_ground_truth.tsv", sep="\t")

test_s1 = pd.read_csv(DATA_DIR / "test/test_source1.tsv", sep="\t")
test_s2 = pd.read_csv(DATA_DIR / "test/test_source2.tsv", sep="\t")
test_s3 = pd.read_csv(DATA_DIR / "test/test_source3.tsv", sep="\t")

print("Shapes:")
print(f"Train S1: {train_s1.shape}")
print(f"Train S2: {train_s2.shape}")
print(f"Train S3: {train_s3.shape}")
print(f"Ground Truth: {gt.shape}")
print(f"Test  S1: {test_s1.shape}")
print(f"Test  S2: {test_s2.shape}")
print(f"Test  S3: {test_s3.shape}")

Shapes:
Train S1: (2206821, 4)
Train S2: (5034616, 4)
Train S3: (5285603, 4)
Ground Truth: (2206821, 2)
Test  S1: (1732544, 4)
Test  S2: (4887273, 4)
Test  S3: (5082316, 4)


In [3]:
print(train_s1.columns.tolist())
print(train_s1['entity_id'].str[:3].value_counts())
print(train_s1.isnull().sum())
print(gt.head(10))

['entity_id', 'business_name', 'business_address', 'country']
entity_id
S1-    2206821
Name: count, dtype: int64
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64
  source1_entity_id                                 matched_entity_ids
0         S1-965667  S2-681193310,S2-743505751,S3-775321672,S3-1129...
1       S1-55344266  S2-249013014,S2-197070651,S3-478195123,S3-3843...
2      S1-343815751             S2-790675320,S2-479876582,S3-878454467
3      S1-656753428              S2-153058913,S2-24659151,S3-679606215
4      S1-102811957  S2-478959098,S2-553508714,S2-625774905,S3-7280...
5       S1-18727616  S2-755677256,S3-187831601,S3-641489370,S3-4762...
6      S1-318373630                          S2-660036492,S3-804600254
7       S1-86989137                          S3-274817120,S3-312496301
8       S1-29845983                          S2-648035184,S3-588502663
9      S1-789009573              S2-383871912,S3-74481402,S3-576451439


In [5]:
# Record counts
print("=== Record Counts ===")
print(f"Train S1: {len(train_s1):,}")
print(f"Train S2: {len(train_s2):,}")
print(f"Train S3: {len(train_s3):,}")
print(f"Test  S1: {len(test_s1):,}")
print(f"Test  S2: {len(test_s2):,}")
print(f"Test  S3: {len(test_s3):,}")

# Country distribution
print("\n=== Country Distribution (Train) ===")
print("S1:\n", train_s1['country'].value_counts(dropna=False))
print("\nS2:\n", train_s2['country'].value_counts(dropna=False))
print("\nS3:\n", train_s3['country'].value_counts(dropna=False))

print("\n=== Country Distribution (Test) ===")
print("S1:\n", test_s1['country'].value_counts(dropna=False))
# Note: France should appear here

=== Record Counts ===
Train S1: 2,206,821
Train S2: 5,034,616
Train S3: 5,285,603
Test  S1: 1,732,544
Test  S2: 4,887,273
Test  S3: 5,082,316

=== Country Distribution (Train) ===
S1:
 country
US       1323633
India     883188
Name: count, dtype: int64

S2:
 country
US       3016817
India    2017799
Name: count, dtype: int64

S3:
 country
US       3170056
India    2115547
Name: count, dtype: int64

=== Country Distribution (Test) ===
S1:
 country
India     809986
US        663106
France    259452
Name: count, dtype: int64


In [6]:
# Parse matched_entity_ids
gt['matched_list'] = gt['matched_entity_ids'].fillna('').apply(
    lambda x: [i.strip() for i in x.split(',') if i.strip()]
)
gt['num_matches'] = gt['matched_list'].apply(len)
gt['is_singleton'] = gt['num_matches'] == 0

print("=== Match Statistics ===")
print(f"Total Source 1 entities: {len(gt):,}")
print(f"Singletons (no matches): {gt['is_singleton'].sum():,} ({gt['is_singleton'].mean()*100:.1f}%)")
print(f"Entities with ≥1 match: {(~gt['is_singleton']).sum():,}")
print("\nMatches per entity distribution:")
print(gt['num_matches'].value_counts().sort_index())

print("\nAverage matches (excluding singletons):", 
      gt.loc[~gt['is_singleton'], 'num_matches'].mean())

=== Match Statistics ===
Total Source 1 entities: 2,206,821
Singletons (no matches): 123,247 (5.6%)
Entities with ≥1 match: 2,083,574

Matches per entity distribution:
num_matches
0     123247
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64

Average matches (excluding singletons): 3.66599170463828


In [7]:
# Parse matched_entity_ids
gt['matched_list'] = gt['matched_entity_ids'].fillna('').apply(
    lambda x: [i.strip() for i in x.split(',') if i.strip()]
)
gt['num_matches'] = gt['matched_list'].apply(len)
gt['is_singleton'] = gt['num_matches'] == 0

print("=== Match Statistics ===")
print(f"Total Source 1 entities: {len(gt):,}")
print(f"Singletons (no matches): {gt['is_singleton'].sum():,} ({gt['is_singleton'].mean()*100:.1f}%)")
print(f"Entities with ≥1 match: {(~gt['is_singleton']).sum():,}")
print("\nMatches per entity distribution:")
print(gt['num_matches'].value_counts().sort_index())

print("\nAverage matches (excluding singletons):", 
      gt.loc[~gt['is_singleton'], 'num_matches'].mean())

=== Match Statistics ===
Total Source 1 entities: 2,206,821
Singletons (no matches): 123,247 (5.6%)
Entities with ≥1 match: 2,083,574

Matches per entity distribution:
num_matches
0     123247
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64

Average matches (excluding singletons): 3.66599170463828


In [8]:
# Length distributions
train_s1['name_len'] = train_s1['business_name'].str.len()
train_s1['addr_len'] = train_s1['business_address'].str.len()

print("Name length stats (S1):")
print(train_s1['name_len'].describe())

# Most common tokens / suffixes
from collections import Counter
import re

def get_tokens(series):
    tokens = []
    for text in series.dropna():
        tokens.extend(re.findall(r'\b\w+\b', text.lower()))
    return Counter(tokens)

name_tokens = get_tokens(train_s1['business_name'])
print("\nTop 40 name tokens:")
print(name_tokens.most_common(40))

# Legal suffixes to watch
suffixes = ['inc', 'llc', 'ltd', 'limited', 'corp', 'corporation', 'pvt', 'private', 
            'co', 'company', 'llp', 'plc', 'gmbh', 'sa', 'sas', 'bv', 'ag']

Name length stats (S1):
count    2.206821e+06
mean     2.403440e+01
std      7.740533e+00
min      3.000000e+00
25%      1.800000e+01
50%      2.400000e+01
75%      3.000000e+01
max      1.050000e+02
Name: name_len, dtype: float64

Top 40 name tokens:
[('limited', 522340), ('private', 432503), ('llc', 355736), ('inc', 238309), ('ltd', 148599), ('pvt', 121462), ('india', 61241), ('and', 57296), ('s', 55791), ('c', 54031), ('care', 53136), ('l', 49410), ('of', 44497), ('associates', 42985), ('group', 40520), ('llp', 39726), ('center', 35433), ('partners', 35084), ('corp', 34792), ('services', 31242), ('p', 29137), ('health', 28264), ('d', 28059), ('co', 27692), ('clinic', 27586), ('pc', 27166), ('solutions', 26415), ('global', 25611), ('trading', 22806), ('international', 20843), ('brothers', 20712), ('technologies', 20620), ('pllc', 20531), ('industries', 18789), ('enterprises', 17902), ('medicine', 17769), ('foundation', 17759), ('ventures', 17198), ('lp', 16563), ('consultants', 15892